# 9.3 Specifying Data Using Composite Data Structures

[Section 9.2](9_2_data_in_dicts.ipynb) showed how to use separate Python dictionaries for each AMPL parameter. This works well, but it means that each food item's properties are scattered across multiple dictionaries. An alternative is to store all attributes of an entity together in a single tuple, producing a more compact and self-contained data definition. This approach is common in Python when working with records or named structures, and it translates naturally to AMPL via dictionary comprehensions that extract each attribute as needed.

In [1]:
# install dependencies
%pip install -q amplpy pandas numpy polars

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


We use the same diet model as in previous sections.

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;
set LINKS within (NUTR cross FOOD);

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];
param amt {NUTR,FOOD} >= 0;

Overwriting diet.mod


In [3]:
ampl.read("diet.mod")

## 9.3.1 Grouping Related Attributes into Tuples

The key idea is to replace the separate `cost`, `f_min`, and `f_max` dictionaries with a single dictionary `foods` whose values are tuples `(cost, f_min, f_max)`. Each entry then contains everything there is to know about one food item. The nutrient data is handled similarly: each entry in `nutrs` holds the pair `(n_min, n_max)` for one nutrient.

Once the composite dictionaries are defined, dictionary comprehensions decompose them into the individual parameter dictionaries that AMPL expects. The comprehension for `cost`, for instance, iterates over `foods.items()`, unpacks each tuple value into its three components, and keeps only the first:

In [6]:
# Define a dictionary with FOOD information for f_min and f_max
foods = {
    "BEEF": (3.59, 2, 10),                  
    "CHK": (2.59, 2, 10),
    "FISH": (2.29, 2, 10),
    "HAM": (2.89, 2, 10),
    "MCH": (1.89, 2, 10),
    "MTL": (1.99, 2, 10),
    "SPG": (1.99, 2, 10),
    "TUR": (2.49, 2, 10)}
        
ampl.set['FOOD'] = list(foods.keys())   # Define the cost parameter for each food item
        
nutrs = {                               # Define nutrient ranges
    "A":    (700, 20000),
    "C":    (700, 20000),
    "B1":   (700, 20000),
    "B2":   (700, 20000),
    "NA":   (0,   50000),
    "CAL":  (16000, 24000),}

ampl.set['NUTR'] = list(nutrs.keys())
ampl.param["cost"] = cost
        
# Map specific attributes to AMPL parameters:
ampl.param["cost"]  = {food:  cost for food, (cost, _, _)  in foods.items()}# Extract 'cost'
ampl.param["f_min"] = {food: f_min for food, (_, f_min, _) in foods.items()}# Extract 'f_min'
ampl.param["f_max"] = {food: f_max for food, (_, _, f_max) in foods.items()}# Extract 'f_max'

ampl.param["n_min"] = {nutr: n_min for nutr, (n_min, _) in nutrs.items()}   # Extract 'n_min'
ampl.param["n_max"] = {nutr: n_max for nutr, (_, n_max) in nutrs.items()}   # Extract 'n_max'
amt = {
    ('A', 'BEEF'):  60, ('C','BEEF'):   20, ('B1','BEEF'):  10, ('B2','BEEF'):  15,
    ('NA','BEEF'):  928, ('CAL','BEEF'): 295, ('A','CHK'):    8, ('B1','CHK'):   20,
    ('B2','CHK'):   20, ('NA','CHK'):   2180, ('CAL','CHK'):  770, ('A','FISH'):   8,
    ('C','FISH'):   10, ('B1','FISH'):  15, ('B2','FISH'):  10, ('NA','FISH'):  945,
    ('CAL','FISH'): 440, ('A','HAM'):    40,('C','HAM'):    40, ('B1','HAM'):   35,
    ('B2','HAM'):   10, ('NA','HAM'):   278, ('CAL','HAM'):  430, ('A','MCH'):    15,
    ('C','MCH'):    35, ('B1','MCH'):   15, ('B2','MCH'):   15, ('NA','MCH'):   1182,
    ('CAL','MCH'):  315, ('A','MTL'):    70, ('C','MTL'):    30, ('B1','MTL'):   15,
    ('B2','MTL'):   15, ('NA','MTL'):   896, ('CAL','MTL'):  400, ('A','SPG'):    25,
    ('C','SPG'):    50, ('B1','SPG'):   25, ('B2','SPG'):   15, ('NA','SPG'):   1329,
    ('CAL','SPG'):  379, ('A','TUR'):    60, ('C','TUR'):    20, ('B1','TUR'):   15,
    ('B2','TUR'):   10, ('NA','TUR'):   1397, ('CAL','TUR'):  450,}
        
ampl.set['LINKS'] = set(amt.keys())     # Set the 'LINKS' set in AMPL using the keys from the 'amt' dictionary (nutrient-food pairs)
ampl.param['amt'] = amt                 # Assign the nutrient amounts for each food item in AMPL

The composite approach keeps the data definition compact: adding a new food item requires a single line in the `foods` dictionary, and all three AMPL parameters are updated automatically when the comprehensions are re-evaluated. The tuple unpacking syntax `(cost, f_min, f_max)` makes the role of each component explicit and guards against accidental misalignment.

The same technique applies whenever multiple AMPL parameters share the same index set and their values belong together conceptually. In larger models it is also common to use Python `dataclasses` or `namedtuples` instead of plain tuples, which gives each field a descriptive name and eliminates the need to remember positional order.